<a href="https://colab.research.google.com/github/studiousdoston/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Menu Detector!")

Menu Detector!


In [7]:
# Import Libraries
from google.colab import drive
import torchvision.transforms as transforms
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset
import os
import numpy as np


In [3]:
drive.mount("/content/drive")

Mounted at /content/drive


In [9]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print("DATASET_PATH:", DATASET_PATH)


CUSTOM_CLASS_MAPPING = {
    "kebab": "kebab",
    "hot_dog" : "hot_dog",
    "baklava": "dessert",     # label grouping | class consoldation
    "cheesecake ": "dessert", # label grouping | class consoldation
    "pilaf": "pilaf",
    "pizza": "pizza",
    "bibimbap": "bibimbap",
}

CLASSES = ["pizza", "bibimbap", "hot_dog", "dessert", "kebab", "pilaf"]
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print(NUM_CLASSES)
print(CLASS_TO_IDX)

transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.299, 0.224, 0.225])
])

# 0~255
# 0.0~1.0
# H, W, C => C, H, W | RGB
# pixel = (pixel - mean) / std

DATASET_PATH: /content/drive/MyDrive/food101_dataset
6
{'pizza': 0, 'bibimbap': 1, 'hot_dog': 2, 'dessert': 3, 'kebab': 4, 'pilaf': 5}


Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.299, 0.224, 0.225])
)

# Custom Dataset Class

In [10]:
from PIL.Image import UnidentifiedImageError
class FoodDataset(Dataset):
  def __init__(self, images, labels, transform=None):
    self.images = images
    self.labels = labels
    self.transform = transform

  def __len__(self):
    print("images_length", len(self.images))
    return len(self.images)

  def __getitem__(self, idx):
    img_path = self.images[idx]
    print("images_path", img_path)
    label = self.labels[idx]
    print("label", label)
    try:
      image = Image.open(img_path).convert("RGB")
    except (UnidentifiedImageError, OSError):
      print(f"Skipping borken image: {img_path}")
      return self.__getitem__((idx+1) % len(self.images))
    if self.transform:
      image = self.transform(image)
    return image, label

# Gather and Split Data

In [12]:
all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
  class_path = os.path.join(DATASET_PATH, original_class)
  print("class_path \n", class_path)
  if not os.path.exists(class_path):
    print(f"Warning: {class_path} not found")
    continue

  for img in os.listdir(class_path):
    if img.endswith(('.jpg', 'jpeg', '.png')):
      full_path = os.path.join(class_path, img)
      all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

print("all_images", all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

class_path 
 /content/drive/MyDrive/food101_dataset/kebab
class_path 
 /content/drive/MyDrive/food101_dataset/hot_dog
class_path 
 /content/drive/MyDrive/food101_dataset/baklava
class_path 
 /content/drive/MyDrive/food101_dataset/cheesecake 
class_path 
 /content/drive/MyDrive/food101_dataset/pilaf
class_path 
 /content/drive/MyDrive/food101_dataset/pizza
class_path 
 /content/drive/MyDrive/food101_dataset/bibimbap
all_images [('/content/drive/MyDrive/food101_dataset/hot_dog/2337278.jpg', 2), ('/content/drive/MyDrive/food101_dataset/pizza/2631321.jpg', 0), ('/content/drive/MyDrive/food101_dataset/pizza/768276.jpg', 0), ('/content/drive/MyDrive/food101_dataset/kebab/kebab_040.jpg', 4), ('/content/drive/MyDrive/food101_dataset/pizza/2321465.jpg', 0), ('/content/drive/MyDrive/food101_dataset/pizza/2218680.jpg', 0), ('/content/drive/MyDrive/food101_dataset/baklava/3172169.jpg', 3), ('/content/drive/MyDrive/food101_dataset/hot_dog/1226430.jpg', 2), ('/content/drive/MyDrive/food101_dataset/p